# Control Variables - 01 Distance to the nearest city
30/05/2026, Kuba Kowalski 

In [6]:
import importlib
import subprocess
import sys
from pathlib import Path

packages = ["geopandas", "pandas", "numpy", "shapely", "scikit-learn", "openpyxl"]

for package in packages:
    try:
        importlib.import_module(package.replace("scikit-learn", "sklearn"))
        print(f"{package}: already installed")
    except ImportError:
        print(f"{package}: installing")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from sklearn.neighbors import BallTree

geopandas: already installed
pandas: already installed
numpy: already installed
shapely: already installed
scikit-learn: already installed
openpyxl: already installed


In [7]:
# Paths - IN
province_polygons = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"
)

cities_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\1_distance-to-the-nearest-city\Africa_City_Populations_Dataset.xlsx"
)

cities_sheet = "Full sample"


# Paths - OUT
out_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\1_distance_to_nearest_city"
)
out_dir.mkdir(parents=True, exist_ok=True)

In [8]:
# Load & inspect

# Load provinces
provinces = gpd.read_file(province_polygons)

print("Province columns:")
print(provinces.columns)

print("\nProvince CRS:")
print(provinces.crs)

zone_id = "GEOLEVEL1" # Prov identifier

if zone_id not in provinces.columns:
    raise ValueError(f"Column '{zone_id}' not found in provinces file.")


# Load cities
cities = pd.read_excel(
    cities_file,
    sheet_name=cities_sheet
)

required_city_cols = ["city_name", "latitude", "longitude", "avg1900"]

missing_cols = [col for col in required_city_cols if col not in cities.columns]
if missing_cols:
    raise ValueError(f"Missing required city columns: {missing_cols}")

Province columns:
Index(['country_id', 'country', 'GEOLEVEL1', 'province', 'cohort',
       'primary_educ', 'higher_educ', 'tertiary_educ', 'n', 'geometry'],
      dtype='object')

Province CRS:
EPSG:4326


In [9]:
# Clean city data
cities["avg1900"] = pd.to_numeric(cities["avg1900"], errors="coerce")
cities["latitude"] = pd.to_numeric(cities["latitude"], errors="coerce")
cities["longitude"] = pd.to_numeric(cities["longitude"], errors="coerce")

cities = cities.dropna(subset=["city_name", "latitude", "longitude", "avg1900"])

# Select cities with population > 5,000 in 1900
cities_1900 = cities[cities["avg1900"] > 5000].copy()

if len(cities_1900) == 0:
    raise ValueError("No cities found with avg1900 > 5000.")

print(f"\nCities with avg1900 > 5000: {len(cities_1900)}")

# Convert cities to GeoDataFrame
cities_gdf = gpd.GeoDataFrame(
    cities_1900,
    geometry=[
        Point(xy) for xy in zip(cities_1900["longitude"], cities_1900["latitude"])
    ],
    crs="EPSG:4326"
)

# Ensure provinces are in WGS84 for final coordinate handling
if provinces.crs is None:
    raise ValueError("Province CRS is missing. Define CRS before running this script.")

provinces_wgs84 = provinces.to_crs("EPSG:4326")


Cities with avg1900 > 5000: 107


In [10]:
# Clean province geometries
provinces = provinces.copy()
provinces["geometry"] = provinces.geometry.make_valid()

provinces = provinces[
    provinces.geometry.notna() &
    ~provinces.geometry.is_empty
].copy()

# Calculate province centroids
africa_equal_area = (
    "+proj=aea +lat_1=-18 +lat_2=21 +lat_0=0 +lon_0=20 "
    "+x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
)

provinces_projected = provinces.to_crs(africa_equal_area)
centroids_projected = provinces_projected.geometry.centroid

centroids_gdf = gpd.GeoDataFrame(
    provinces[[zone_id]].copy(),
    geometry=centroids_projected,
    crs=africa_equal_area
).to_crs("EPSG:4326")

centroids_gdf["centroid_latitude"] = centroids_gdf.geometry.y
centroids_gdf["centroid_longitude"] = centroids_gdf.geometry.x

# Drop invalid centroids
bad_centroids = centroids_gdf[
    centroids_gdf[["centroid_latitude", "centroid_longitude"]]
    .isna()
    .any(axis=1)
]

if len(bad_centroids) > 0:
    print("Dropping invalid centroids:")
    print(bad_centroids[[zone_id, "centroid_latitude", "centroid_longitude"]])

centroids_gdf = centroids_gdf[
    centroids_gdf[["centroid_latitude", "centroid_longitude"]]
    .notna()
    .all(axis=1)
].copy()

# Clean city coordinates
cities_gdf = cities_gdf[
    cities_gdf[["latitude", "longitude"]]
    .notna()
    .all(axis=1)
].copy()

# Nearest city calculation using haversine distance (shortest possible distance between 2 points on the surface of a sphere)
earth_radius_km = 6371.0088

city_coords_rad = np.radians(
    cities_gdf[["latitude", "longitude"]].to_numpy()
)

province_coords_rad = np.radians(
    centroids_gdf[["centroid_latitude", "centroid_longitude"]].to_numpy()
)

tree = BallTree(city_coords_rad, metric="haversine")

dist_rad, nearest_idx = tree.query(province_coords_rad, k=1)

dist_km = dist_rad.flatten() * earth_radius_km
nearest_idx = nearest_idx.flatten()

nearest_cities = cities_gdf.iloc[nearest_idx].reset_index(drop=True)

distance_to_city = pd.DataFrame({
    zone_id: centroids_gdf[zone_id].values,
    "1a_distance-to-the-nearest-city": dist_km,
    "1b_name-of-the-nearest-city": nearest_cities["city_name"].values
})

print("\nOutput preview:")
print(distance_to_city.head())


Output preview:
  GEOLEVEL1  1a_distance-to-the-nearest-city 1b_name-of-the-nearest-city
0    204001                       315.741128                      Sokoto
1    204002                       373.385872                      Iseyin
2    204003                        42.290667                  Porto Novo
3    204004                       221.362007                      Iseyin
4    204005                       107.035656                      Abomey


In [11]:
# Export CSV
csv_path = out_dir / "1_distance_to_nearest_city_1900.csv"
distance_to_city.to_csv(csv_path, index=False)
print(f"\nCSV saved to: {csv_path}")


# Re-join to spatial province file
provinces_distance_city = provinces.merge(
    distance_to_city,
    on=zone_id,
    how="left"
)

# Export spatial file
gpkg_path = out_dir / "1_distance_to_nearest_city_1900.gpkg"
provinces_distance_city.to_file(
    gpkg_path,
    driver="GPKG"
)
print(f"GPKG saved to: {gpkg_path}")


# Sanity checks
print("\nMissing values:")
print(distance_to_city.isna().sum())

print("\nShortest distances:")
print(distance_to_city.sort_values("1a_distance-to-the-nearest-city").head(10))

print("\nLongest distances:")
print(distance_to_city.sort_values("1a_distance-to-the-nearest-city", ascending=False).head(10))


CSV saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\1_distance_to_nearest_city\1_distance_to_nearest_city_1900.csv
GPKG saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\1_distance_to_nearest_city\1_distance_to_nearest_city_1900.gpkg

Missing values:
GEOLEVEL1                          0
1a_distance-to-the-nearest-city    0
1b_name-of-the-nearest-city        0
dtype: int64

Shortest distances:
     GEOLEVEL1  1a_distance-to-the-nearest-city 1b_name-of-the-nearest-city
251     854003                         2.717149                 Ouagadougou
247     854003                         2.717149                 Ouagadougou
249     854003                         2.717149                 Ouagadougou
250     854003                         2.717149                 Ouagadougou
245     854003                         2.717149                 Ouagadougou
248     854003                       

In [12]:
# Joys of visualization

import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import gaussian_kde

# ---- paths ----
africa_outline_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"

map_output_dir = out_dir / "maps"
map_output_dir.mkdir(exist_ok=True)

# ---- load Africa outline ----
africa = gpd.read_file(africa_outline_file)

# Use the final province layer with distance variable attached
provinces_plot_base = provinces_distance_city.copy()

if africa.crs != provinces_plot_base.crs:
    africa = africa.to_crs(provinces_plot_base.crs)

# ---- variables to map ----
variables = {
    "1a_distance-to-the-nearest-city": "Distance to nearest city with >5,000 inhabitants in 1900"
}

for var, label in variables.items():

    gdf_plot = provinces_plot_base[provinces_plot_base[var].notna()].copy()

    values = gdf_plot[var].dropna().values

    if len(values) == 0:
        print(f"Skipping {var}: no valid values")
        continue

    vmin = values.min()
    vmax = values.max()

    fig, ax = plt.subplots(figsize=(16, 20))

    # ---- Africa background outline ----
    africa.plot(
        ax=ax,
        facecolor="none",
        edgecolor="lightgrey",
        linewidth=0.5
    )

    # ---- province layer ----
    gdf_plot.plot(
        column=var,
        cmap="viridis",
        linewidth=0.4,
        edgecolor="black",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        ax=ax
    )

    ax.set_title(label)
    ax.set_axis_off()

    # ---- fixed Africa-wide extent ----
    minx, miny, maxx, maxy = africa.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    # ---- horizontal colorbar ----
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.04
    )

    cbar.set_label("Kilometers")

    # ---- wave axis above colorbar ----
    cb_pos = cbar.ax.get_position()

    wave_ax = fig.add_axes([
        cb_pos.x0,
        cb_pos.y1 + 0.005,
        cb_pos.width,
        0.05
    ])

    mean_val = values.mean()
    min_val = values.min()
    max_val = values.max()

    fig.text(
        0.5,
        0.02,
        f"Mean: {mean_val:.3f} km   Min: {min_val:.3f} km   Max: {max_val:.3f} km",
        ha="center",
        fontsize=10
    )

    if len(values) > 1 and vmin < vmax:
        kde = gaussian_kde(values)
        x = np.linspace(vmin, vmax, 300)
        y = kde(x)

        wave_ax.plot(x, y, linewidth=1.5)
        wave_ax.fill_between(x, y, alpha=0.25)

    wave_ax.set_xlim(vmin, vmax)
    wave_ax.set_xticks([])
    wave_ax.set_yticks([])

    for spine in wave_ax.spines.values():
        spine.set_visible(False)

    # ---- save ----
    safe_var_name = var.replace("-", "_")
    out_file = map_output_dir / f"{safe_var_name}_map.png"

    plt.savefig(
        out_file,
        dpi=800,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Saved: {out_file}")

print("Done.")

Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\1_distance_to_nearest_city\maps\1a_distance_to_the_nearest_city_map.png
Done.


In [13]:
# second visualization for laughs

cities_plot = cities_gdf.copy()

if cities_plot.crs != provinces_plot_base.crs:
    cities_plot = cities_plot.to_crs(provinces_plot_base.crs)

for var, label in variables.items():

    gdf_plot = provinces_plot_base[provinces_plot_base[var].notna()].copy()
    values = gdf_plot[var].dropna().values

    if len(values) == 0:
        print(f"Skipping {var}: no valid values")
        continue

    vmin = values.min()
    vmax = values.max()

    fig, ax = plt.subplots(figsize=(16, 20))

    africa.plot(
        ax=ax,
        facecolor="none",
        edgecolor="lightgrey",
        linewidth=0.5
    )

    gdf_plot.plot(
        column=var,
        cmap="viridis",
        linewidth=0.4,
        edgecolor="black",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        ax=ax
    )

    cities_plot.plot(
        ax=ax,
        color="red",
        markersize=8,
        alpha=0.75,
        edgecolor="black",
        linewidth=0.2
    )

    ax.set_title(label + "\nCity points: population > 5,000 in 1900")
    ax.set_axis_off()

    minx, miny, maxx, maxy = africa.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.04
    )
    cbar.set_label("Kilometers")

    safe_var_name = var.replace("-", "_")
    out_file = map_output_dir / f"{safe_var_name}_map_with_city_points.png"

    plt.savefig(
        out_file,
        dpi=800,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Saved: {out_file}")

Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\1_distance_to_nearest_city\maps\1a_distance_to_the_nearest_city_map_with_city_points.png


# Control Variables - 02 Elevation
30/05/2026, Kuba Kowalski 

Bounding box data req: Xmin = -19.4765657186508	  Ymin = -37.76029790849866	  Xmax = 54.773438572883634	  Ymax = 36.55553077578038

Bounding box data req:-19.5, -37.8, 54.8, 36.6


In [14]:
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterio.mask import mask
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

province_polygons = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"
)

elevation_raster = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\2_elevation\wc2.1_10m_elev\wc2.1_10m_elev.tif"
)

out_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\2_elevation"
)
out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

zone_id = "GEOLEVEL1"
control_var = "2_elevation-sd"


In [15]:
# ------------------------------------------------------------------
# LOAD PROVINCES
# ------------------------------------------------------------------

provinces = gpd.read_file(province_polygons)

if provinces.crs is None:
    raise ValueError("Province file has no CRS.")

if zone_id not in provinces.columns:
    raise ValueError(f"Column '{zone_id}' not found in province file.")

# ------------------------------------------------------------------
# CLEAN PROVINCES
# ------------------------------------------------------------------

# Needed to prevent NULL values in GEOLEVEL1 072002, 072003, 072004, 204008

from shapely.geometry import Polygon, MultiPolygon, GeometryCollection
from shapely.ops import unary_union

def keep_polygonal_geometry(geom):
    if geom is None or geom.is_empty:
        return None

    if isinstance(geom, (Polygon, MultiPolygon)):
        return geom

    if isinstance(geom, GeometryCollection):
        polys = [
            g for g in geom.geoms
            if isinstance(g, (Polygon, MultiPolygon)) and not g.is_empty
        ]

        if len(polys) == 0:
            return None

        return unary_union(polys)

    return None


provinces = provinces.copy()
provinces[zone_id] = provinces[zone_id].astype(str).str.strip().str.zfill(6)

# Use GeoPandas method, not Shapely geometry method
provinces["geometry"] = provinces.geometry.make_valid()

# Keep only polygonal geometries
provinces["geometry"] = provinces.geometry.apply(keep_polygonal_geometry)

bad = provinces[
    provinces.geometry.isna() |
    provinces.geometry.is_empty
].copy()

if len(bad) > 0:
    print("Dropping bad geometries:")
    print(bad[[zone_id, "country"]].drop_duplicates())

provinces = provinces[
    provinces.geometry.notna() &
    ~provinces.geometry.is_empty
].copy()

# Raster controls should be one row per province
provinces = (
    provinces
    .sort_values(zone_id)
    .drop_duplicates(subset=[zone_id])
    .copy()
)

print("Valid unique provinces after cleaning:", len(provinces))

# ------------------------------------------------------------------
# CLEAN AND DISSOLVE PROVINCES BEFORE RASTER EXTRACTION
# ------------------------------------------------------------------

provinces = provinces.copy()
provinces[zone_id] = provinces[zone_id].astype(str).str.strip().str.zfill(6)

provinces["geometry"] = provinces.geometry.make_valid()

provinces = provinces[
    provinces.geometry.notna() &
    ~provinces.geometry.is_empty
].copy()

# Dissolve all cohort duplicates into one geometry per province
provinces = provinces.dissolve(
    by=zone_id,
    as_index=False
)

provinces["geometry"] = provinces.geometry.make_valid()

provinces = provinces[
    provinces.geometry.notna() &
    ~provinces.geometry.is_empty
].copy()

print("Unique dissolved provinces:", len(provinces))

print(
    provinces[
        provinces[zone_id].isin(["072002", "072003", "072004", "204008"])
    ][[zone_id, "geometry"]]
)

Dropping bad geometries:
    GEOLEVEL1   country
392    231017  ethiopia
Valid unique provinces after cleaning: 304
Unique dissolved provinces: 304
   GEOLEVEL1                                           geometry
6     072002  POLYGON ((27.56286 -21.19583, 27.56278 -21.204...
7     072003  POLYGON ((25.69257 -25.23868, 25.68849 -25.240...
8     072004  POLYGON ((27.82842 -21.93943, 27.8637 -21.9549...
40    204008  POLYGON ((2.44572 6.39741, 2.44227 6.38806, 2....


In [16]:
# ------------------------------------------------------------------
# CALCULATE ELEVATION SD PER PROVINCE
# ------------------------------------------------------------------
from shapely.geometry import mapping

results = []

with rasterio.open(elevation_raster) as src:

    print("Raster CRS:", src.crs)
    print("Raster nodata:", src.nodata)
    print("Raster bounds:", src.bounds)
    print("Raster resolution:", src.res)

    if provinces.crs != src.crs:
        provinces = provinces.to_crs(src.crs)

    nodata = src.nodata

    for idx, row in provinces.iterrows():

        province_id = row[zone_id]
        geom = [mapping(row.geometry)]

        try:
            out_image, out_transform = mask(
                src,
                geom,
                crop=True,
                filled=True,
                nodata=nodata,
                all_touched=True
            )

            values = out_image[0].astype("float64")

            if nodata is not None:
                values = values[values != nodata]

            values = values[np.isfinite(values)]

            # WorldClim elevation is in meters.
            # Remove implausible values as an extra safety filter.
            values = values[(values > -500) & (values < 9000)]

            if len(values) == 0:
                elev_sd = np.nan
                elev_mean = np.nan
                elev_min = np.nan
                elev_max = np.nan
                n_pixels = 0
            else:
                elev_sd = np.std(values, ddof=0)
                elev_mean = np.mean(values)
                elev_min = np.min(values)
                elev_max = np.max(values)
                n_pixels = len(values)

        except Exception as e:
            print(f"Failed for {province_id}: {e}")
            elev_sd = np.nan
            elev_mean = np.nan
            elev_min = np.nan
            elev_max = np.nan
            n_pixels = 0

        results.append({
            zone_id: province_id,
            control_var: elev_sd,
            "2b_elevation-mean": elev_mean,
            "2c_elevation-min": elev_min,
            "2d_elevation-max": elev_max,
            "2e_elevation-valid-pixels": n_pixels
        })

Raster CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Raster nodata: -32768.0
Raster bounds: BoundingBox(left=-180.0, bottom=-90.0, right=180.0, top=90.0)
Raster resolution: (0.16666666666666666, 0.16666666666666666)


In [17]:
# ------------------------------------------------------------------
# EXPORT TABLE
# ------------------------------------------------------------------

elevation_df = pd.DataFrame(results)

csv_path = out_dir / "2_elevation_sd_worldclim.csv"
elevation_df.to_csv(csv_path, index=False)

print(f"CSV saved to: {csv_path}")

# ------------------------------------------------------------------
# JOIN BACK TO PROVINCES
# ------------------------------------------------------------------

provinces_elevation = provinces.merge(
    elevation_df,
    on=zone_id,
    how="left"
)

gpkg_path = out_dir / "2_elevation_sd_worldclim.gpkg"
provinces_elevation.to_file(gpkg_path, driver="GPKG")

print(f"GPKG saved to: {gpkg_path}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nMissing values:")
print(elevation_df.isna().sum())

print("\nSummary statistics:")
print(elevation_df[control_var].describe())

print("\nLowest elevation SD:")
print(elevation_df.sort_values(control_var).head(10))

print("\nHighest elevation SD:")
print(elevation_df.sort_values(control_var, ascending=False).head(10))

print("Done.")

CSV saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\2_elevation\2_elevation_sd_worldclim.csv
GPKG saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\2_elevation\2_elevation_sd_worldclim.gpkg

Missing values:
GEOLEVEL1                    0
2_elevation-sd               0
2b_elevation-mean            0
2c_elevation-min             0
2d_elevation-max             0
2e_elevation-valid-pixels    0
dtype: int64

Summary statistics:
count    304.000000
mean     150.200139
std      147.260198
min        0.000000
25%       49.199616
50%       97.574037
75%      207.662931
max      785.772585
Name: 2_elevation-sd, dtype: float64

Lowest elevation SD:
    GEOLEVEL1  2_elevation-sd  2b_elevation-mean  2c_elevation-min  \
40     204008        0.000000           4.000000               4.0   
225    800102        2.449490        1172.000000            1169.0   
224    800101        5.253570     

In [18]:
print(provinces_elevation.columns.tolist())

['GEOLEVEL1', 'geometry', 'country_id', 'country', 'province', 'cohort', 'primary_educ', 'higher_educ', 'tertiary_educ', 'n', '2_elevation-sd', '2b_elevation-mean', '2c_elevation-min', '2d_elevation-max', '2e_elevation-valid-pixels']


In [19]:
# visualization - elevation SD

import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import gaussian_kde

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

africa_outline_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"

map_output_dir = out_dir / "maps"
map_output_dir.mkdir(exist_ok=True)

# ------------------------------------------------------------------
# LOAD AFRICA OUTLINE
# ------------------------------------------------------------------

africa = gpd.read_file(africa_outline_file)

if africa.crs != provinces_elevation.crs:
    africa = africa.to_crs(provinces_elevation.crs)

# ------------------------------------------------------------------
# VARIABLES TO MAP
# ------------------------------------------------------------------

variables = {
    "2_elevation-sd": "Standard deviation of elevation"
}

for var, label in variables.items():

    gdf_plot = provinces_elevation[provinces_elevation[var].notna()].copy()
    values = gdf_plot[var].dropna().values

    if len(values) == 0:
        print(f"Skipping {var}: no valid values")
        continue

    vmin = values.min()
    vmax = values.max()

    fig, ax = plt.subplots(figsize=(16, 20))

    africa.plot(
        ax=ax,
        facecolor="none",
        edgecolor="lightgrey",
        linewidth=0.5
    )

    gdf_plot.plot(
        column=var,
        cmap="viridis",
        linewidth=0.4,
        edgecolor="black",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        ax=ax
    )

    ax.set_title(label)
    ax.set_axis_off()

    minx, miny, maxx, maxy = africa.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.04
    )

    cbar.set_label("Elevation SD, meters")

    cb_pos = cbar.ax.get_position()

    wave_ax = fig.add_axes([
        cb_pos.x0,
        cb_pos.y1 + 0.005,
        cb_pos.width,
        0.05
    ])

    mean_val = values.mean()
    min_val = values.min()
    max_val = values.max()

    fig.text(
        0.5,
        0.02,
        f"Mean: {mean_val:.3f} m   Min: {min_val:.3f} m   Max: {max_val:.3f} m",
        ha="center",
        fontsize=10
    )

    if len(values) > 1 and vmin < vmax:
        kde = gaussian_kde(values)
        x = np.linspace(vmin, vmax, 300)
        y = kde(x)

        wave_ax.plot(x, y, linewidth=1.5)
        wave_ax.fill_between(x, y, alpha=0.25)

    wave_ax.set_xlim(vmin, vmax)
    wave_ax.set_xticks([])
    wave_ax.set_yticks([])

    for spine in wave_ax.spines.values():
        spine.set_visible(False)

    out_file = map_output_dir / f"{var.replace('-', '_')}_map.png"

    plt.savefig(
        out_file,
        dpi=800,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Saved: {out_file}")

print("Done.")

Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\2_elevation\maps\2_elevation_sd_map.png
Done.


# Control Variables - 03 Rain
30/05/2026, Kuba Kowalski 

In [20]:
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterio.mask import mask
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

province_polygons = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"
)

precip_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\3_rain\wc2.1_10m_prec"
)

out_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\3_rain"
)
out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

zone_id = "GEOLEVEL1"
control_var = "3_average-annual-precipitation-mm"

In [21]:
# ------------------------------------------------------------------
# LOAD PRECIPITATION FILES
# ------------------------------------------------------------------

precip_files = [
    precip_dir / f"wc2.1_10m_prec_{month:02d}.tif"
    for month in range(1, 13)
]

missing_files = [fp for fp in precip_files if not fp.exists()]

if missing_files:
    raise FileNotFoundError(
        "Missing monthly precipitation files:\n"
        + "\n".join(str(fp) for fp in missing_files)
    )

print("Monthly precipitation files:")
for fp in precip_files:
    print(fp.name)

# ------------------------------------------------------------------
# LOAD PROVINCES
# ------------------------------------------------------------------

provinces = gpd.read_file(province_polygons)

if provinces.crs is None:
    raise ValueError("Province file has no CRS.")

if zone_id not in provinces.columns:
    raise ValueError(f"Column '{zone_id}' not found in province file.")

# ------------------------------------------------------------------
# CHECK RASTER METADATA
# ------------------------------------------------------------------

with rasterio.open(precip_files[0]) as src0:
    raster_crs = src0.crs
    raster_transform = src0.transform
    raster_shape = src0.shape
    raster_nodata = src0.nodata

    print("Raster CRS:", raster_crs)
    print("Raster nodata:", raster_nodata)
    print("Raster bounds:", src0.bounds)
    print("Raster resolution:", src0.res)
    print("Raster shape:", raster_shape)

# Reproject provinces to raster CRS
if provinces.crs != raster_crs:
    provinces = provinces.to_crs(raster_crs)

Monthly precipitation files:
wc2.1_10m_prec_01.tif
wc2.1_10m_prec_02.tif
wc2.1_10m_prec_03.tif
wc2.1_10m_prec_04.tif
wc2.1_10m_prec_05.tif
wc2.1_10m_prec_06.tif
wc2.1_10m_prec_07.tif
wc2.1_10m_prec_08.tif
wc2.1_10m_prec_09.tif
wc2.1_10m_prec_10.tif
wc2.1_10m_prec_11.tif
wc2.1_10m_prec_12.tif
Raster CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Raster nodata: -32768.0
Raster bounds: BoundingBox(left=-180.0, bottom=-90.0, right=180.0, top=90.0)
Raster resolution: (0.16666666666666666, 0.16666666666666666)
Raster shape: (1080, 2160)


In [22]:
# ------------------------------------------------------------------
# CREATE TEMPORARY ANNUAL PRECIPITATION RASTER
# ------------------------------------------------------------------

annual_precip_file = out_dir / "worldclim_annual_precipitation_mm.tif"

print("Creating annual precipitation raster...")

with rasterio.open(precip_files[0]) as src0:
    meta = src0.meta.copy()
    meta.update({
        "dtype": "float32",
        "count": 1,
        "nodata": -9999
    })

    annual_sum = None
    valid_count = None

    for fp in precip_files:
        with rasterio.open(fp) as src:
            if src.crs != raster_crs:
                raise ValueError(f"CRS mismatch in {fp.name}")

            if src.shape != raster_shape:
                raise ValueError(f"Shape mismatch in {fp.name}")

            if src.transform != raster_transform:
                raise ValueError(f"Transform mismatch in {fp.name}")

            arr = src.read(1).astype("float32")
            nodata = src.nodata

            valid = np.isfinite(arr)

            if nodata is not None:
                valid = valid & (arr != nodata)

            # WorldClim precipitation is in mm/month.
            # Negative precipitation is invalid.
            valid = valid & (arr >= 0)

            if annual_sum is None:
                annual_sum = np.zeros(arr.shape, dtype="float32")
                valid_count = np.zeros(arr.shape, dtype="uint8")

            annual_sum[valid] += arr[valid]
            valid_count[valid] += 1

    # Only keep pixels valid for all 12 months
    annual_sum[valid_count < 12] = -9999

    with rasterio.open(annual_precip_file, "w", **meta) as dst:
        dst.write(annual_sum, 1)

print(f"Annual precipitation raster saved to: {annual_precip_file}")

# ------------------------------------------------------------------
# CALCULATE PROVINCE-LEVEL AVERAGE ANNUAL PRECIPITATION
# ------------------------------------------------------------------

results = []

with rasterio.open(annual_precip_file) as src:

    nodata = src.nodata

    for idx, row in provinces.iterrows():

        province_id = row[zone_id]
        geom = [row.geometry]

        try:
            out_image, out_transform = mask(
                src,
                geom,
                crop=True,
                filled=True,
                nodata=nodata,
                all_touched=True
            )

            values = out_image[0].astype("float64")

            if nodata is not None:
                values = values[values != nodata]

            values = values[np.isfinite(values)]

            # Safety filter: annual precipitation cannot be negative
            values = values[values >= 0]

            if len(values) == 0:
                mean_precip = np.nan
                min_precip = np.nan
                max_precip = np.nan
                sd_precip = np.nan
                n_pixels = 0
            else:
                mean_precip = np.mean(values)
                min_precip = np.min(values)
                max_precip = np.max(values)
                sd_precip = np.std(values, ddof=0)
                n_pixels = len(values)

        except Exception as e:
            print(f"Failed for {province_id}: {e}")
            mean_precip = np.nan
            min_precip = np.nan
            max_precip = np.nan
            sd_precip = np.nan
            n_pixels = 0

        results.append({
            zone_id: province_id,
            control_var: mean_precip,
            "3b_annual-precipitation-min-mm": min_precip,
            "3c_annual-precipitation-max-mm": max_precip,
            "3d_annual-precipitation-sd-mm": sd_precip,
            "3e_annual-precipitation-valid-pixels": n_pixels
        })

Creating annual precipitation raster...
Annual precipitation raster saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\3_rain\worldclim_annual_precipitation_mm.tif
Failed for 231017: argument of type 'NoneType' is not iterable
Failed for 231017: argument of type 'NoneType' is not iterable
Failed for 231017: argument of type 'NoneType' is not iterable
Failed for 231017: argument of type 'NoneType' is not iterable
Failed for 231017: argument of type 'NoneType' is not iterable
Failed for 231017: argument of type 'NoneType' is not iterable
Failed for 231017: argument of type 'NoneType' is not iterable


In [23]:
# ------------------------------------------------------------------
# EXPORT TABLE
# ------------------------------------------------------------------

precip_df = pd.DataFrame(results)

csv_path = out_dir / "3_average_annual_precipitation_worldclim.csv"
precip_df.to_csv(csv_path, index=False)

print(f"CSV saved to: {csv_path}")

# ------------------------------------------------------------------
# JOIN BACK TO PROVINCES
# ------------------------------------------------------------------

provinces_precip = provinces.merge(
    precip_df,
    on=zone_id,
    how="left"
)

gpkg_path = out_dir / "3_average_annual_precipitation_worldclim.gpkg"
provinces_precip.to_file(gpkg_path, driver="GPKG")

print(f"GPKG saved to: {gpkg_path}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nMissing values:")
print(precip_df.isna().sum())

print("\nSummary statistics:")
print(precip_df[control_var].describe())

print("\nLowest average annual precipitation:")
print(precip_df.sort_values(control_var).head(10))

print("\nHighest average annual precipitation:")
print(precip_df.sort_values(control_var, ascending=False).head(10))

print("Done.")

CSV saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\3_rain\3_average_annual_precipitation_worldclim.csv
GPKG saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\3_rain\3_average_annual_precipitation_worldclim.gpkg

Missing values:
GEOLEVEL1                               0
3_average-annual-precipitation-mm       7
3b_annual-precipitation-min-mm          7
3c_annual-precipitation-max-mm          7
3d_annual-precipitation-sd-mm           7
3e_annual-precipitation-valid-pixels    0
dtype: int64

Summary statistics:
count    2118.000000
mean     1174.868407
std       671.680570
min        20.937021
25%       763.805243
50%      1056.107143
75%      1320.677419
max      3547.142857
Name: 3_average-annual-precipitation-mm, dtype: float64

Lowest average annual precipitation:
     GEOLEVEL1  3_average-annual-precipitation-mm  \
1882    729011                          20.937021   
1881   

In [24]:
# Joys of visualization - average annual precipitation

import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import gaussian_kde

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

africa_outline_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"

map_output_dir = out_dir / "maps"
map_output_dir.mkdir(exist_ok=True)

# ------------------------------------------------------------------
# LOAD OUTPUTS IF NEEDED
# ------------------------------------------------------------------

precip_gpkg = out_dir / "3_average_annual_precipitation_worldclim.gpkg"

if "provinces_precip" not in globals():
    provinces_precip = gpd.read_file(precip_gpkg)

africa = gpd.read_file(africa_outline_file)

if africa.crs != provinces_precip.crs:
    africa = africa.to_crs(provinces_precip.crs)

# ------------------------------------------------------------------
# VARIABLES TO MAP
# ------------------------------------------------------------------

print("Available columns:")
print(provinces_precip.columns.tolist())

variables = {
    "3_average-annual-precipitation-mm": "Average annual precipitation"
}

for var, label in variables.items():

    if var not in provinces_precip.columns:
        raise ValueError(f"Column not found: {var}")

    gdf_plot = provinces_precip[provinces_precip[var].notna()].copy()
    values = gdf_plot[var].dropna().values

    if len(values) == 0:
        print(f"Skipping {var}: no valid values")
        continue

    vmin = values.min()
    vmax = values.max()

    fig, ax = plt.subplots(figsize=(16, 20))

    africa.plot(
        ax=ax,
        facecolor="none",
        edgecolor="lightgrey",
        linewidth=0.5
    )

    gdf_plot.plot(
        column=var,
        cmap="viridis",
        linewidth=0.4,
        edgecolor="black",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        ax=ax
    )

    ax.set_title(label)
    ax.set_axis_off()

    minx, miny, maxx, maxy = africa.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.04
    )

    cbar.set_label("Millimeters per year")

    cb_pos = cbar.ax.get_position()

    wave_ax = fig.add_axes([
        cb_pos.x0,
        cb_pos.y1 + 0.005,
        cb_pos.width,
        0.05
    ])

    mean_val = values.mean()
    min_val = values.min()
    max_val = values.max()

    fig.text(
        0.5,
        0.02,
        f"Mean: {mean_val:.3f} mm/year   Min: {min_val:.3f} mm/year   Max: {max_val:.3f} mm/year",
        ha="center",
        fontsize=10
    )

    if len(values) > 1 and vmin < vmax:
        kde = gaussian_kde(values)
        x = np.linspace(vmin, vmax, 300)
        y = kde(x)

        wave_ax.plot(x, y, linewidth=1.5)
        wave_ax.fill_between(x, y, alpha=0.25)

    wave_ax.set_xlim(vmin, vmax)
    wave_ax.set_xticks([])
    wave_ax.set_yticks([])

    for spine in wave_ax.spines.values():
        spine.set_visible(False)

    out_file = map_output_dir / f"{var.replace('-', '_')}_worldclim_map.png"

    plt.savefig(
        out_file,
        dpi=800,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Saved: {out_file}")

print("Done.")

Available columns:
['country_id', 'country', 'GEOLEVEL1', 'province', 'cohort', 'primary_educ', 'higher_educ', 'tertiary_educ', 'n', 'geometry', '3_average-annual-precipitation-mm', '3b_annual-precipitation-min-mm', '3c_annual-precipitation-max-mm', '3d_annual-precipitation-sd-mm', '3e_annual-precipitation-valid-pixels']
Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\3_rain\maps\3_average_annual_precipitation_mm_worldclim_map.png
Done.


# Control Variables - 04 Temperature
30/05/2026, Kuba Kowalski 

In [25]:
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterio.mask import mask
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

province_polygons = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"
)

temp_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\4_temperature\wc2.1_10m_tavg"
)

out_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\4_temperature"
)
out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

zone_id = "GEOLEVEL1"
control_var = "4_average-annual-temperature-c"

In [26]:
# ------------------------------------------------------------------
# LOAD TEMPERATURE FILES
# ------------------------------------------------------------------

temp_files = [
    temp_dir / f"wc2.1_10m_tavg_{month:02d}.tif"
    for month in range(1, 13)
]

missing_files = [fp for fp in temp_files if not fp.exists()]

if missing_files:
    raise FileNotFoundError(
        "Missing monthly temperature files:\n"
        + "\n".join(str(fp) for fp in missing_files)
    )

print("Monthly temperature files:")
for fp in temp_files:
    print(fp.name)

# ------------------------------------------------------------------
# LOAD PROVINCES
# ------------------------------------------------------------------

provinces = gpd.read_file(province_polygons)

if provinces.crs is None:
    raise ValueError("Province file has no CRS.")

if zone_id not in provinces.columns:
    raise ValueError(f"Column '{zone_id}' not found in province file.")

# ------------------------------------------------------------------
# CHECK RASTER METADATA
# ------------------------------------------------------------------

with rasterio.open(temp_files[0]) as src0:
    raster_crs = src0.crs
    raster_transform = src0.transform
    raster_shape = src0.shape
    raster_nodata = src0.nodata

    print("Raster CRS:", raster_crs)
    print("Raster nodata:", raster_nodata)
    print("Raster bounds:", src0.bounds)
    print("Raster resolution:", src0.res)
    print("Raster shape:", raster_shape)

if provinces.crs != raster_crs:
    provinces = provinces.to_crs(raster_crs)

Monthly temperature files:
wc2.1_10m_tavg_01.tif
wc2.1_10m_tavg_02.tif
wc2.1_10m_tavg_03.tif
wc2.1_10m_tavg_04.tif
wc2.1_10m_tavg_05.tif
wc2.1_10m_tavg_06.tif
wc2.1_10m_tavg_07.tif
wc2.1_10m_tavg_08.tif
wc2.1_10m_tavg_09.tif
wc2.1_10m_tavg_10.tif
wc2.1_10m_tavg_11.tif
wc2.1_10m_tavg_12.tif
Raster CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Raster nodata: -3.3999999521443642e+38
Raster bounds: BoundingBox(left=-180.0, bottom=-90.0, right=180.0, top=90.0)
Raster resolution: (0.16666666666666666, 0.16666666666666666)
Raster shape: (1080, 2160)


In [27]:
# ------------------------------------------------------------------
# CREATE ANNUAL MEAN TEMPERATURE RASTER
# ------------------------------------------------------------------

annual_temp_file = out_dir / "worldclim_annual_mean_temperature_c.tif"

print("Creating annual mean temperature raster...")

with rasterio.open(temp_files[0]) as src0:
    meta = src0.meta.copy()
    meta.update({
        "dtype": "float32",
        "count": 1,
        "nodata": -9999
    })

    temp_sum = None
    valid_count = None

    for fp in temp_files:
        with rasterio.open(fp) as src:
            if src.crs != raster_crs:
                raise ValueError(f"CRS mismatch in {fp.name}")

            if src.shape != raster_shape:
                raise ValueError(f"Shape mismatch in {fp.name}")

            if src.transform != raster_transform:
                raise ValueError(f"Transform mismatch in {fp.name}")

            arr = src.read(1).astype("float32")
            nodata = src.nodata

            valid = np.isfinite(arr)

            if nodata is not None:
                valid = valid & (arr != nodata)

            # WorldClim tavg is normally in °C.
            # This safety filter removes impossible or corrupted values.
            valid = valid & (arr > -100) & (arr < 100)

            if temp_sum is None:
                temp_sum = np.zeros(arr.shape, dtype="float32")
                valid_count = np.zeros(arr.shape, dtype="uint8")

            temp_sum[valid] += arr[valid]
            valid_count[valid] += 1

    annual_mean_temp = np.full(temp_sum.shape, -9999, dtype="float32")
    valid_all_months = valid_count == 12
    annual_mean_temp[valid_all_months] = temp_sum[valid_all_months] / 12

    with rasterio.open(annual_temp_file, "w", **meta) as dst:
        dst.write(annual_mean_temp, 1)

print(f"Annual mean temperature raster saved to: {annual_temp_file}")

# ------------------------------------------------------------------
# CALCULATE PROVINCE-LEVEL AVERAGE ANNUAL TEMPERATURE
# ------------------------------------------------------------------

results = []

with rasterio.open(annual_temp_file) as src:

    nodata = src.nodata

    for idx, row in provinces.iterrows():

        province_id = row[zone_id]
        geom = [row.geometry]

        try:
            out_image, out_transform = mask(
                src,
                geom,
                crop=True,
                filled=True,
                nodata=nodata,
                all_touched=True
            )

            values = out_image[0].astype("float64")

            if nodata is not None:
                values = values[values != nodata]

            values = values[np.isfinite(values)]
            values = values[(values > -100) & (values < 100)]

            if len(values) == 0:
                mean_temp = np.nan
                min_temp = np.nan
                max_temp = np.nan
                sd_temp = np.nan
                n_pixels = 0
            else:
                mean_temp = np.mean(values)
                min_temp = np.min(values)
                max_temp = np.max(values)
                sd_temp = np.std(values, ddof=0)
                n_pixels = len(values)

        except Exception as e:
            print(f"Failed for {province_id}: {e}")
            mean_temp = np.nan
            min_temp = np.nan
            max_temp = np.nan
            sd_temp = np.nan
            n_pixels = 0

        results.append({
            zone_id: province_id,
            control_var: mean_temp,
            "4b_annual-temperature-min-c": min_temp,
            "4c_annual-temperature-max-c": max_temp,
            "4d_annual-temperature-sd-c": sd_temp,
            "4e_annual-temperature-valid-pixels": n_pixels
        })


Creating annual mean temperature raster...
Annual mean temperature raster saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\4_temperature\worldclim_annual_mean_temperature_c.tif
Failed for 231017: argument of type 'NoneType' is not iterable
Failed for 231017: argument of type 'NoneType' is not iterable
Failed for 231017: argument of type 'NoneType' is not iterable
Failed for 231017: argument of type 'NoneType' is not iterable
Failed for 231017: argument of type 'NoneType' is not iterable
Failed for 231017: argument of type 'NoneType' is not iterable
Failed for 231017: argument of type 'NoneType' is not iterable


In [28]:
# ------------------------------------------------------------------
# EXPORT TABLE
# ------------------------------------------------------------------

temp_df = pd.DataFrame(results)

csv_path = out_dir / "4_average_annual_temperature_worldclim.csv"
temp_df.to_csv(csv_path, index=False)

print(f"CSV saved to: {csv_path}")

# ------------------------------------------------------------------
# JOIN BACK TO PROVINCES
# ------------------------------------------------------------------

provinces_temp = provinces.merge(
    temp_df,
    on=zone_id,
    how="left"
)

gpkg_path = out_dir / "4_average_annual_temperature_worldclim.gpkg"
provinces_temp.to_file(gpkg_path, driver="GPKG")

print(f"GPKG saved to: {gpkg_path}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nMissing values:")
print(temp_df.isna().sum())

print("\nSummary statistics:")
print(temp_df[control_var].describe())

print("\nLowest average annual temperature:")
print(temp_df.sort_values(control_var).head(10))

print("\nHighest average annual temperature:")
print(temp_df.sort_values(control_var, ascending=False).head(10))

print("Done.")

CSV saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\4_temperature\4_average_annual_temperature_worldclim.csv
GPKG saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\4_temperature\4_average_annual_temperature_worldclim.gpkg

Missing values:
GEOLEVEL1                             0
4_average-annual-temperature-c        7
4b_annual-temperature-min-c           7
4c_annual-temperature-max-c           7
4d_annual-temperature-sd-c            7
4e_annual-temperature-valid-pixels    0
dtype: int64

Summary statistics:
count    2118.000000
mean       24.198406
std         2.930807
min        15.713979
25%        21.719407
50%        24.576536
75%        26.717170
max        29.206382
Name: 4_average-annual-temperature-c, dtype: float64

Lowest average annual temperature:
    GEOLEVEL1  4_average-annual-temperature-c  4b_annual-temperature-min-c  \
382    231014                       15.713

In [29]:
# Joys of visualization - average annual temperature

import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import gaussian_kde

africa_outline_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"

map_output_dir = out_dir / "maps"
map_output_dir.mkdir(exist_ok=True)

temp_gpkg = out_dir / "4_average_annual_temperature_worldclim.gpkg"

if "provinces_temp" not in globals():
    provinces_temp = gpd.read_file(temp_gpkg)

africa = gpd.read_file(africa_outline_file)

if africa.crs != provinces_temp.crs:
    africa = africa.to_crs(provinces_temp.crs)

print("Available columns:")
print(provinces_temp.columns.tolist())

variables = {
    "4_average-annual-temperature-c": "Average annual temperature"
}

for var, label in variables.items():

    if var not in provinces_temp.columns:
        raise ValueError(f"Column not found: {var}")

    gdf_plot = provinces_temp[provinces_temp[var].notna()].copy()
    values = gdf_plot[var].dropna().values

    if len(values) == 0:
        print(f"Skipping {var}: no valid values")
        continue

    vmin = values.min()
    vmax = values.max()

    fig, ax = plt.subplots(figsize=(16, 20))

    africa.plot(
        ax=ax,
        facecolor="none",
        edgecolor="lightgrey",
        linewidth=0.5
    )

    gdf_plot.plot(
        column=var,
        cmap="viridis",
        linewidth=0.4,
        edgecolor="black",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        ax=ax
    )

    ax.set_title(label)
    ax.set_axis_off()

    minx, miny, maxx, maxy = africa.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.04
    )

    cbar.set_label("Degrees Celsius")

    cb_pos = cbar.ax.get_position()

    wave_ax = fig.add_axes([
        cb_pos.x0,
        cb_pos.y1 + 0.005,
        cb_pos.width,
        0.05
    ])

    mean_val = values.mean()
    min_val = values.min()
    max_val = values.max()

    fig.text(
        0.5,
        0.02,
        f"Mean: {mean_val:.3f} °C   Min: {min_val:.3f} °C   Max: {max_val:.3f} °C",
        ha="center",
        fontsize=10
    )

    if len(values) > 1 and vmin < vmax:
        kde = gaussian_kde(values)
        x = np.linspace(vmin, vmax, 300)
        y = kde(x)

        wave_ax.plot(x, y, linewidth=1.5)
        wave_ax.fill_between(x, y, alpha=0.25)

    wave_ax.set_xlim(vmin, vmax)
    wave_ax.set_xticks([])
    wave_ax.set_yticks([])

    for spine in wave_ax.spines.values():
        spine.set_visible(False)

    out_file = map_output_dir / f"{var.replace('-', '_')}_worldclim_map.png"

    plt.savefig(
        out_file,
        dpi=800,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Saved: {out_file}")

print("Done.")

Available columns:
['country_id', 'country', 'GEOLEVEL1', 'province', 'cohort', 'primary_educ', 'higher_educ', 'tertiary_educ', 'n', 'geometry', '4_average-annual-temperature-c', '4b_annual-temperature-min-c', '4c_annual-temperature-max-c', '4d_annual-temperature-sd-c', '4e_annual-temperature-valid-pixels']
Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\4_temperature\maps\4_average_annual_temperature_c_worldclim_map.png
Done.


# Control Variables - 06 River
30/05/2026, Kuba Kowalski 

Definition: 1 if a main river polyline intersects/crosses the province polygon, 0 otherwise

In [30]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

province_polygons = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"
)

rivers_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\6_river\ne_50m_rivers_lake_centerlines\ne_50m_rivers_lake_centerlines.shp"
)

out_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\6_river"
)
out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

zone_id = "GEOLEVEL1"
control_var = "6_river"

In [31]:
# ------------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------------

provinces = gpd.read_file(province_polygons)
rivers = gpd.read_file(rivers_file)

if provinces.crs is None:
    raise ValueError("Province file has no CRS.")

if rivers.crs is None:
    raise ValueError("River file has no CRS.")

if zone_id not in provinces.columns:
    raise ValueError(f"Column '{zone_id}' not found in province file.")

# Match CRS
if rivers.crs != provinces.crs:
    rivers = rivers.to_crs(provinces.crs)

# Optional: repair invalid geometries
provinces["geometry"] = provinces.geometry.make_valid()
rivers["geometry"] = rivers.geometry.make_valid()

# Remove empty geometries
provinces = provinces[provinces.geometry.notna() & ~provinces.geometry.is_empty].copy()
rivers = rivers[rivers.geometry.notna() & ~rivers.geometry.is_empty].copy()

print(f"Provinces: {len(provinces)}")
print(f"River features: {len(rivers)}")

Provinces: 2118
River features: 477


In [32]:
# ------------------------------------------------------------------
# SPATIAL JOIN: PROVINCES INTERSECTING RIVERS
# ------------------------------------------------------------------

intersections = gpd.sjoin(
    provinces[[zone_id, "geometry"]],
    rivers[["geometry"]],
    how="left",
    predicate="intersects"
)

river_presence = (
    intersections
    .groupby(zone_id)["index_right"]
    .apply(lambda x: int(x.notna().any()))
    .reset_index(name=control_var)
)

# ------------------------------------------------------------------
# JOIN BACK TO PROVINCES
# ------------------------------------------------------------------

provinces_river = provinces.merge(
    river_presence,
    on=zone_id,
    how="left"
)

provinces_river[control_var] = (
    provinces_river[control_var]
    .fillna(0)
    .astype(int)
)

In [33]:
# ------------------------------------------------------------------
# EXPORT TABLE
# ------------------------------------------------------------------

river_df = provinces_river[[zone_id, control_var]].copy()

csv_path = out_dir / "6_main_river_dummy.csv"
river_df.to_csv(csv_path, index=False)

print(f"CSV saved to: {csv_path}")

# ------------------------------------------------------------------
# EXPORT SPATIAL FILE
# ------------------------------------------------------------------

gpkg_path = out_dir / "6_main_river_dummy.gpkg"
provinces_river.to_file(gpkg_path, driver="GPKG")

print(f"GPKG saved to: {gpkg_path}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nRiver dummy counts:")
print(river_df[control_var].value_counts(dropna=False))

print("\nShare of provinces with main river:")
print(river_df[control_var].mean())

print("Done.")

CSV saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\6_river\6_main_river_dummy.csv
GPKG saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\6_river\6_main_river_dummy.gpkg

River dummy counts:
6_river
0    1325
1     793
Name: count, dtype: int64

Share of provinces with main river:
0.37440982058545796
Done.


In [34]:
# vis

import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

africa_outline_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"

rivers_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\6_river\ne_50m_rivers_lake_centerlines\ne_50m_rivers_lake_centerlines.shp"
)

map_output_dir = out_dir / "maps"
map_output_dir.mkdir(exist_ok=True)

# ------------------------------------------------------------------
# LOAD OUTPUTS IF NEEDED
# ------------------------------------------------------------------

river_gpkg = out_dir / "6_main_river_dummy.gpkg"

if "provinces_river" not in globals():
    provinces_river = gpd.read_file(river_gpkg)

africa = gpd.read_file(africa_outline_file)
rivers = gpd.read_file(rivers_file)

if africa.crs != provinces_river.crs:
    africa = africa.to_crs(provinces_river.crs)

if rivers.crs != provinces_river.crs:
    rivers = rivers.to_crs(provinces_river.crs)

# ------------------------------------------------------------------
# MAP
# ------------------------------------------------------------------

var = "6_river"

fig, ax = plt.subplots(figsize=(16, 20))

africa.plot(
    ax=ax,
    facecolor="none",
    edgecolor="lightgrey",
    linewidth=0.5
)

provinces_river.plot(
    column=var,
    cmap="viridis",
    linewidth=0.4,
    edgecolor="black",
    legend=True,
    categorical=True,
    ax=ax
)

rivers.plot(
    ax=ax,
    color="blue",
    linewidth=0.5,
    alpha=0.7
)

ax.set_title("Main river dummy by province")
ax.set_axis_off()

minx, miny, maxx, maxy = africa.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

out_file = map_output_dir / "6_main_river_dummy_map.png"

plt.savefig(
    out_file,
    dpi=800,
    bbox_inches="tight"
)

plt.close()

print(f"Saved: {out_file}")

Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\6_river\maps\6_main_river_dummy_map.png


# Control Variables - 07 Sea
30/05/2026, Kuba Kowalski 

Definition: 1 if a province polygon intersects/is adjacent to the ocean polygon, 0 otherwise

In [35]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

province_polygons = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"
)

ocean_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\7_sea\ne_10m_ocean\ne_10m_ocean.shp"
)

out_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\7_sea"
)

out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

zone_id = "GEOLEVEL1"
control_var = "7_sea"

In [36]:
# ------------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------------

provinces = gpd.read_file(province_polygons)
ocean = gpd.read_file(ocean_file)

if zone_id not in provinces.columns:
    raise ValueError(f"Column '{zone_id}' not found.")

if provinces.crs is None:
    raise ValueError("Province CRS missing.")

if ocean.crs is None:
    raise ValueError("Ocean CRS missing.")

# ------------------------------------------------------------------
# CRS
# ------------------------------------------------------------------

if ocean.crs != provinces.crs:
    ocean = ocean.to_crs(provinces.crs)

# ------------------------------------------------------------------
# GEOMETRY CLEANING
# ------------------------------------------------------------------

provinces["geometry"] = provinces.geometry.make_valid()
ocean["geometry"] = ocean.geometry.make_valid()

provinces = provinces[
    provinces.geometry.notna() &
    ~provinces.geometry.is_empty
].copy()

ocean = ocean[
    ocean.geometry.notna() &
    ~ocean.geometry.is_empty
].copy()

print(f"Provinces: {len(provinces)}")
print(f"Ocean polygons: {len(ocean)}")


Provinces: 2118
Ocean polygons: 1


In [37]:
# ------------------------------------------------------------------
# CREATE 15 KM PROVINCE BUFFER
# ------------------------------------------------------------------

buffer_crs = "EPSG:6933"
buffer_distance_m = 15000

provinces_buffer = provinces.to_crs(buffer_crs).copy()
provinces_buffer["geometry"] = provinces_buffer.geometry.buffer(buffer_distance_m)
provinces_buffer = provinces_buffer.to_crs(provinces.crs)

buffer_file = out_dir / "7_province_buffer_15km.geojson"

provinces_buffer.to_file(
    buffer_file,
    driver="GeoJSON"
)

print(f"Buffered provinces saved to: {buffer_file}")

# ------------------------------------------------------------------
# MATCH CRS
# ------------------------------------------------------------------

if ocean.crs != provinces_buffer.crs:
    ocean = ocean.to_crs(provinces_buffer.crs)

# ------------------------------------------------------------------
# SPATIAL JOIN: BUFFERED PROVINCES INTERSECTING OCEAN
# ------------------------------------------------------------------

intersections = gpd.sjoin(
    provinces_buffer[[zone_id, "geometry"]],
    ocean[["geometry"]],
    how="left",
    predicate="intersects"
)

sea_presence = (
    intersections
    .groupby(zone_id)["index_right"]
    .apply(lambda x: int(x.notna().any()))
    .reset_index(name=control_var)
)

# ------------------------------------------------------------------
# JOIN RESULT BACK TO ORIGINAL, UNBUFFERED PROVINCES
# ------------------------------------------------------------------

provinces_sea = provinces.merge(
    sea_presence,
    on=zone_id,
    how="left"
)

provinces_sea[control_var] = (
    provinces_sea[control_var]
    .fillna(0)
    .astype(int)
)

Buffered provinces saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\7_sea\7_province_buffer_15km.geojson


In [38]:
# ------------------------------------------------------------------
# EXPORT CSV
# ------------------------------------------------------------------

sea_df = provinces_sea[[zone_id, control_var]].copy()

csv_path = out_dir / "7_sea_dummy.csv"
sea_df.to_csv(csv_path, index=False)

print(f"CSV saved to: {csv_path}")

# ------------------------------------------------------------------
# EXPORT GPKG
# ------------------------------------------------------------------

gpkg_path = out_dir / "7_sea_dummy.gpkg"

provinces_sea.to_file(
    gpkg_path,
    driver="GPKG"
)

print(f"GPKG saved to: {gpkg_path}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nSea dummy counts:")
print(sea_df[control_var].value_counts(dropna=False))

print("\nShare coastal provinces:")
print(sea_df[control_var].mean())

print("Done.")

CSV saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\7_sea\7_sea_dummy.csv
GPKG saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\7_sea\7_sea_dummy.gpkg

Sea dummy counts:
7_sea
0    1719
1     399
Name: count, dtype: int64

Share coastal provinces:
0.18838526912181303
Done.


In [39]:
# vis

import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

africa_outline_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"

ocean_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\7_sea\ne_10m_ocean\ne_10m_ocean.shp"
)

map_output_dir = out_dir / "maps"
map_output_dir.mkdir(exist_ok=True)

# ------------------------------------------------------------------
# LOAD OUTPUTS IF NEEDED
# ------------------------------------------------------------------

sea_gpkg = out_dir / "7_sea_dummy.gpkg"

if "provinces_sea" not in globals():
    provinces_sea = gpd.read_file(sea_gpkg)

africa = gpd.read_file(africa_outline_file)
ocean = gpd.read_file(ocean_file)

if africa.crs != provinces_sea.crs:
    africa = africa.to_crs(provinces_sea.crs)

if ocean.crs != provinces_sea.crs:
    ocean = ocean.to_crs(provinces_sea.crs)

# ------------------------------------------------------------------
# MAP
# ------------------------------------------------------------------

var = "7_sea"

fig, ax = plt.subplots(figsize=(16, 20))

ocean.plot(
    ax=ax,
    facecolor="lightblue",
    edgecolor="none",
    alpha=0.35
)

africa.plot(
    ax=ax,
    facecolor="none",
    edgecolor="lightgrey",
    linewidth=0.5
)

provinces_sea.plot(
    column=var,
    cmap="viridis",
    linewidth=0.4,
    edgecolor="black",
    legend=True,
    categorical=True,
    ax=ax
)

ax.set_title("Sea dummy by province")
ax.set_axis_off()

minx, miny, maxx, maxy = africa.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

out_file = map_output_dir / "7_sea_dummy_map.png"

plt.savefig(
    out_file,
    dpi=800,
    bbox_inches="tight"
)

plt.close()

print(f"Saved: {out_file}")

Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\7_sea\maps\7_sea_dummy_map.png


# Control Variables - 08 border
30/05/2026, Kuba Kowalski 

Definition:
1 = province shares a boundary with a province in another country
0 = otherwise

In [40]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

province_polygons = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"
)

countries_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"
)

out_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\8_border"
)
out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

zone_id = "GEOLEVEL1"
province_country_col = "country"
country_outline_col = "CNTRY_NAME"

control_var = "8_border-dummy"

buffer_crs = "EPSG:6933"
boundary_buffer_m = 2000  # tolerance for geometry mismatch

selected_countries = [
    "Benin",
    "Botswana",
    "Burkina Faso",
    "Côte d’Ivoire",
    "Ethiopia",
    "Ghana",
    "Guinea",
    "Kenya",
    "Malawi",
    "Mali",
    "Mozambique",
    "Rwanda",
    "Senegal",
    "Sierra Leone",
    "Tanzania",
    "Togo",
    "Uganda",
    "Zambia",
]

In [41]:
# ------------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------------

def clean_country_name(x):
    if pd.isna(x):
        return np.nan

    x = str(x).lower().strip()
    x = x.replace("’", "'").replace("`", "'").replace("´", "'")

    fixes = {
        "côte d'ivoire": "cote d'ivoire",
        "cote d'ivoire": "cote d'ivoire",
        "ivory coast": "cote d'ivoire",
        "united republic of tanzania": "tanzania",
        "tanzania": "tanzania",
    }

    return fixes.get(x, x)

selected_countries_clean = {clean_country_name(c) for c in selected_countries}

# ------------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------------

provinces_all = gpd.read_file(province_polygons)
countries = gpd.read_file(countries_file)

if provinces_all.crs is None:
    raise ValueError("Province CRS missing.")

if countries.crs is None:
    raise ValueError("Country outline CRS missing.")

for col in [zone_id, province_country_col]:
    if col not in provinces_all.columns:
        raise ValueError(f"Missing province column: {col}")

if country_outline_col not in countries.columns:
    raise ValueError(f"Missing country outline column: {country_outline_col}")

countries = countries.to_crs(provinces_all.crs)

provinces_all["geometry"] = provinces_all.geometry.make_valid()
countries["geometry"] = countries.geometry.make_valid()

provinces_all = provinces_all[
    provinces_all.geometry.notna() &
    ~provinces_all.geometry.is_empty
].copy()

countries = countries[
    countries.geometry.notna() &
    ~countries.geometry.is_empty
].copy()

provinces_all["country_clean"] = provinces_all[province_country_col].apply(clean_country_name)
countries["country_clean"] = countries[country_outline_col].apply(clean_country_name)


In [42]:
# ------------------------------------------------------------------
# UNIQUE PROVINCES ONLY
# ------------------------------------------------------------------

sort_cols = [zone_id]
if "cohort" in provinces_all.columns:
    sort_cols.append("cohort")

provinces = (
    provinces_all
    .sort_values(sort_cols)
    .drop_duplicates(subset=[zone_id])
    .copy()
)

print("Full province-cohort rows:", len(provinces_all))
print("Unique province geometries:", len(provinces))

# ------------------------------------------------------------------
# 1. BORDER WITH ANOTHER SELECTED COUNTRY USING SELECTED PROVINCES
# ------------------------------------------------------------------

provinces_m = provinces.to_crs(buffer_crs).copy()

left = provinces_m[[zone_id, "country_clean"]].copy()
left["geometry"] = provinces_m.geometry.boundary.buffer(boundary_buffer_m)
left = gpd.GeoDataFrame(left, geometry="geometry", crs=buffer_crs)

right = provinces_m[[zone_id, "country_clean", "geometry"]].copy()
right = gpd.GeoDataFrame(right, geometry="geometry", crs=buffer_crs)

selected_matches = gpd.sjoin(
    left,
    right,
    how="inner",
    predicate="intersects",
    lsuffix="left",
    rsuffix="right"
)

selected_matches = selected_matches[
    selected_matches[f"{zone_id}_left"] != selected_matches[f"{zone_id}_right"]
].copy()

selected_matches["selected_country_border"] = (
    selected_matches["country_clean_left"] != selected_matches["country_clean_right"]
)

selected_border_df = (
    selected_matches
    .groupby(f"{zone_id}_left")["selected_country_border"]
    .any()
    .astype(int)
    .reset_index()
)

selected_border_df.columns = [zone_id, "8a_border-with-selected-country"]

# ------------------------------------------------------------------
# 2. BORDER WITH NON-SELECTED COUNTRIES USING COUNTRY OUTLINE BOUNDARIES
# ------------------------------------------------------------------

countries_external = countries[
    ~countries["country_clean"].isin(selected_countries_clean)
].copy()

provinces_m = provinces.to_crs(buffer_crs).copy()
countries_external_m = countries_external.to_crs(buffer_crs).copy()

province_boundaries = provinces_m[[zone_id, "country_clean"]].copy()
province_boundaries["geometry"] = provinces_m.geometry.boundary.buffer(boundary_buffer_m)
province_boundaries = gpd.GeoDataFrame(
    province_boundaries,
    geometry="geometry",
    crs=buffer_crs
)

external_boundaries = countries_external_m[["country_clean"]].copy()
external_boundaries["geometry"] = countries_external_m.geometry.boundary
external_boundaries = gpd.GeoDataFrame(
    external_boundaries,
    geometry="geometry",
    crs=buffer_crs
)

# Fast prefilter: keep only external boundaries near selected province boundaries
province_union = province_boundaries.geometry.union_all()
external_boundaries = external_boundaries[
    external_boundaries.intersects(province_union.buffer(50000))
].copy()

external_matches = gpd.sjoin(
    province_boundaries,
    external_boundaries,
    how="left",
    predicate="intersects",
    lsuffix="province",
    rsuffix="external"
)

external_matches["external_country_border"] = external_matches["index_external"].notna()

external_border_df = (
    external_matches
    .groupby(zone_id)["external_country_border"]
    .any()
    .astype(int)
    .reset_index()
)

external_border_df.columns = [zone_id, "8b_border-with-nonselected-country"]

# ------------------------------------------------------------------
# COMBINE UNIQUE-PROVINCE RESULTS
# ------------------------------------------------------------------

border_df = provinces[[zone_id]].merge(
    selected_border_df,
    on=zone_id,
    how="left"
).merge(
    external_border_df,
    on=zone_id,
    how="left"
)

border_df["8a_border-with-selected-country"] = (
    border_df["8a_border-with-selected-country"]
    .fillna(0)
    .astype(int)
)

border_df["8b_border-with-nonselected-country"] = (
    border_df["8b_border-with-nonselected-country"]
    .fillna(0)
    .astype(int)
)

border_df[control_var] = (
    (border_df["8a_border-with-selected-country"] == 1) |
    (border_df["8b_border-with-nonselected-country"] == 1)
).astype(int)

# ------------------------------------------------------------------
# MERGE BACK TO ALL COHORT ROWS
# ------------------------------------------------------------------

provinces_border = provinces_all.merge(
    border_df,
    on=zone_id,
    how="left"
)

provinces_border[control_var] = (
    provinces_border[control_var]
    .fillna(0)
    .astype(int)
)

Full province-cohort rows: 2118
Unique province geometries: 304


In [43]:
# ------------------------------------------------------------------
# EXPORT CSV
# ------------------------------------------------------------------

csv_path = out_dir / "8_border_dummy_combined.csv"

provinces_border[
    [
        zone_id,
        province_country_col,
        "8a_border-with-selected-country",
        "8b_border-with-nonselected-country",
        control_var
    ]
].to_csv(csv_path, index=False)

print(f"CSV saved to: {csv_path}")

# ------------------------------------------------------------------
# EXPORT GPKG
# ------------------------------------------------------------------

gpkg_path = out_dir / "8_border_dummy_combined.gpkg"

provinces_border.to_file(
    gpkg_path,
    driver="GPKG"
)

print(f"GPKG saved to: {gpkg_path}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nFinal border dummy counts:")
print(provinces_border[control_var].value_counts(dropna=False))

print("\nComponents:")
print(provinces_border[
    [
        "8a_border-with-selected-country",
        "8b_border-with-nonselected-country",
        control_var
    ]
].sum())

print("\nShare border provinces:")
print(provinces_border[control_var].mean())

print("\nSelected countries represented in provinces:")
print(sorted(provinces_border["country_clean"].dropna().unique()))

print("Done.")

CSV saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\8_border\8_border_dummy_combined.csv
GPKG saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\8_border\8_border_dummy_combined.gpkg

Final border dummy counts:
8_border-dummy
1    1428
0     690
Name: count, dtype: int64

Components:
8a_border-with-selected-country       1106
8b_border-with-nonselected-country     882
8_border-dummy                        1428
dtype: int64

Share border provinces:
0.6742209631728046

Selected countries represented in provinces:
['benin', 'botswana', 'burkina_faso', 'cameroon', 'cote_divoire', 'ethiopia', 'ghana', 'guinea', 'kenya', 'liberia', 'malawi', 'mali', 'mozambique', 'rwanda', 'senegal', 'sierra_leone', 'south_sudan', 'sudan', 'tanzania', 'togo', 'uganda', 'zambia', 'zimbabwe']
Done.


In [44]:
# Joys of visualization - border dummy

import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

africa_outline_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"

map_output_dir = out_dir / "maps"
map_output_dir.mkdir(exist_ok=True)

# ------------------------------------------------------------------
# LOAD OUTPUTS IF NEEDED
# ------------------------------------------------------------------

border_gpkg = out_dir / "8_border_dummy_combined.gpkg"

if "provinces_border" not in globals():
    provinces_border = gpd.read_file(border_gpkg)

africa = gpd.read_file(africa_outline_file)

if africa.crs != provinces_border.crs:
    africa = africa.to_crs(provinces_border.crs)

# ------------------------------------------------------------------
# MAP
# ------------------------------------------------------------------

var = "8_border-dummy"

fig, ax = plt.subplots(figsize=(16, 20))

africa.plot(
    ax=ax,
    facecolor="none",
    edgecolor="lightgrey",
    linewidth=0.5
)

provinces_border.plot(
    column=var,
    cmap="viridis",
    linewidth=0.4,
    edgecolor="black",
    legend=True,
    categorical=True,
    ax=ax
)

ax.set_title("International border dummy by province")
ax.set_axis_off()

minx, miny, maxx, maxy = africa.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

out_file = map_output_dir / "8_border_dummy_combined_map.png"

plt.savefig(
    out_file,
    dpi=800,
    bbox_inches="tight"
)

plt.close()

print(f"Saved: {out_file}")

Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\8_border\maps\8_border_dummy_combined_map.png


# Control Variables - 05 area, 09 latitude, 10 longitude
30/05/2026, Kuba Kowalski 



In [45]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

province_polygons = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"
)

out_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\9_lat_lon_area"
)
out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

zone_id = "GEOLEVEL1"

lat_var = "9_latitude"
lon_var = "10_longitude"
area_var = "11_area-km2"

In [46]:
# ------------------------------------------------------------------
# LOAD PROVINCES
# ------------------------------------------------------------------

provinces = gpd.read_file(province_polygons)

if provinces.crs is None:
    raise ValueError("Province CRS missing.")

if zone_id not in provinces.columns:
    raise ValueError(f"Missing column: {zone_id}")

provinces["geometry"] = provinces.geometry.make_valid()

provinces = provinces[
    provinces.geometry.notna() &
    ~provinces.geometry.is_empty
].copy()

# Keep one geometry per province, then merge results back to all cohort rows
provinces_all = provinces.copy()

sort_cols = [zone_id]
if "cohort" in provinces.columns:
    sort_cols.append("cohort")

provinces_unique = (
    provinces
    .sort_values(sort_cols)
    .drop_duplicates(subset=[zone_id])
    .copy()
)

print("Full province-cohort rows:", len(provinces_all))
print("Unique province geometries:", len(provinces_unique))

Full province-cohort rows: 2118
Unique province geometries: 304


In [47]:
# ------------------------------------------------------------------
# CALCULATE AREA AND CENTROID
# ------------------------------------------------------------------

metric_crs = "EPSG:6933"  # Equal-area projection, meters
provinces_metric = provinces_unique.to_crs(metric_crs).copy()

provinces_metric[area_var] = provinces_metric.geometry.area / 1_000_000

centroids_metric = provinces_metric.geometry.centroid

centroids_wgs84 = gpd.GeoSeries(
    centroids_metric,
    crs=metric_crs
).to_crs("EPSG:4326")

provinces_metric[lat_var] = centroids_wgs84.y.values
provinces_metric[lon_var] = centroids_wgs84.x.values

controls_df = provinces_metric[
    [zone_id, lat_var, lon_var, area_var]
].copy()

# ------------------------------------------------------------------
# MERGE BACK TO ALL COHORT ROWS
# ------------------------------------------------------------------

provinces_lat_lon_area = provinces_all.merge(
    controls_df,
    on=zone_id,
    how="left"
)


In [48]:
# ------------------------------------------------------------------
# EXPORT CSV
# ------------------------------------------------------------------

csv_path = out_dir / "9_10_11_latitude_longitude_area.csv"

provinces_lat_lon_area[
    [zone_id, lat_var, lon_var, area_var]
].to_csv(csv_path, index=False)

print(f"CSV saved to: {csv_path}")

# ------------------------------------------------------------------
# EXPORT GPKG
# ------------------------------------------------------------------

gpkg_path = out_dir / "9_10_11_latitude_longitude_area.gpkg"

provinces_lat_lon_area.to_file(
    gpkg_path,
    driver="GPKG"
)

print(f"GPKG saved to: {gpkg_path}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nSummary statistics:")
print(controls_df[[lat_var, lon_var, area_var]].describe())

print("\nLargest provinces:")
print(controls_df.sort_values(area_var, ascending=False).head(10))

print("Done.")

CSV saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\9_lat_lon_area\9_10_11_latitude_longitude_area.csv
GPKG saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\9_lat_lon_area\9_10_11_latitude_longitude_area.gpkg

Summary statistics:
       9_latitude  10_longitude    11_area-km2
count  304.000000    304.000000     304.000000
mean     0.405815     16.987134   37079.780827
std     12.012010     19.686553   66808.121578
min    -25.960710    -17.266914      28.359871
25%     -9.988756     -3.491802    4106.572295
50%      5.053409     27.882929   12456.383261
75%      9.923594     33.609419   44889.330844
max     19.981732     42.172526  640656.445965

Largest provinces:
     GEOLEVEL1  9_latitude  10_longitude    11_area-km2
349     231004    7.247505     41.020752  640656.445965
972     466006   19.981732     -3.571450  501740.563935
1882    729011   19.538131     29.334752  364282

## Control variable - 13 gold

Simple dummy variable. If gold deposit present within province, assign 1, otherwise 0. 

In [49]:
# Definition: 1 if at least one MRDS record with gold in commod1/commod2/commod3 falls within province

import geopandas as gpd
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

province_polygons = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"
)

gold_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\13_gold\mrds\MRDS_global.geojson"
)

out_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\13_gold"
)
out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

zone_id = "GEOLEVEL1"
control_var = "13_gold-deposit"

commodity_cols = ["commod1", "commod2", "commod3"]

In [50]:
# ------------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------------

provinces = gpd.read_file(province_polygons)
mrds = gpd.read_file(gold_file)

if provinces.crs is None:
    raise ValueError("Province file has no CRS.")

if mrds.crs is None:
    raise ValueError("MRDS gold file has no CRS.")

if zone_id not in provinces.columns:
    raise ValueError(f"Column '{zone_id}' not found in province file.")

missing_commodity_cols = [
    col for col in commodity_cols
    if col not in mrds.columns
]

if missing_commodity_cols:
    raise ValueError(f"Missing commodity columns in MRDS file: {missing_commodity_cols}")

# Match CRS
if mrds.crs != provinces.crs:
    mrds = mrds.to_crs(provinces.crs)

# Repair invalid geometries
provinces["geometry"] = provinces.geometry.make_valid()
mrds["geometry"] = mrds.geometry.make_valid()

# Remove empty geometries
provinces = provinces[
    provinces.geometry.notna() &
    ~provinces.geometry.is_empty
].copy()

mrds = mrds[
    mrds.geometry.notna() &
    ~mrds.geometry.is_empty
].copy()

print(f"Province rows: {len(provinces)}")
print(f"MRDS records: {len(mrds)}")

Province rows: 2118
MRDS records: 304613


In [51]:
# ------------------------------------------------------------------
# FILTER MRDS TO GOLD RECORDS
# ------------------------------------------------------------------

gold_mask = False

for col in commodity_cols:
    gold_mask = gold_mask | mrds[col].astype(str).str.contains(
        "gold",
        case=False,
        na=False
    )

gold = mrds[gold_mask].copy()

print(f"Gold records: {len(gold)}")

if len(gold) == 0:
    raise ValueError("No gold records found in commod1/commod2/commod3.")

# ------------------------------------------------------------------
# SPATIAL JOIN: GOLD POINTS WITHIN PROVINCES
# ------------------------------------------------------------------

intersections = gpd.sjoin(
    provinces[[zone_id, "geometry"]],
    gold[["geometry"]],
    how="left",
    predicate="intersects"
)

gold_presence = (
    intersections
    .groupby(zone_id)["index_right"]
    .apply(lambda x: int(x.notna().any()))
    .reset_index(name=control_var)
)

# ------------------------------------------------------------------
# JOIN BACK TO PROVINCES
# ------------------------------------------------------------------

provinces_gold = provinces.merge(
    gold_presence,
    on=zone_id,
    how="left"
)

provinces_gold[control_var] = (
    provinces_gold[control_var]
    .fillna(0)
    .astype(int)
)

Gold records: 84406


In [52]:
# ------------------------------------------------------------------
# EXPORT TABLE
# ------------------------------------------------------------------

gold_df = provinces_gold[[zone_id, control_var]].copy()

csv_path = out_dir / "13_gold_deposit_dummy.csv"
gold_df.to_csv(csv_path, index=False)

print(f"CSV saved to: {csv_path}")

# ------------------------------------------------------------------
# EXPORT SPATIAL FILE
# ------------------------------------------------------------------

gpkg_path = out_dir / "13_gold_deposit_dummy.gpkg"
provinces_gold.to_file(gpkg_path, driver="GPKG")

print(f"GPKG saved to: {gpkg_path}")

# Also export filtered gold points for inspection
gold_points_path = out_dir / "13_gold_records_mrds_filtered.gpkg"
gold.to_file(gold_points_path, driver="GPKG")

print(f"Filtered gold records saved to: {gold_points_path}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nGold dummy counts:")
print(gold_df[control_var].value_counts(dropna=False))

print("\nShare of provinces with gold deposit:")
print(gold_df[control_var].mean())

print("Done.")

CSV saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\13_gold\13_gold_deposit_dummy.csv
GPKG saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\13_gold\13_gold_deposit_dummy.gpkg
Filtered gold records saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\13_gold\13_gold_records_mrds_filtered.gpkg

Gold dummy counts:
13_gold-deposit
0    1763
1     355
Name: count, dtype: int64

Share of provinces with gold deposit:
0.1676109537299339
Done.


In [53]:
# Visualization - gold deposit dummy

import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

africa_outline_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"

map_output_dir = out_dir / "maps"
map_output_dir.mkdir(exist_ok=True)

# ------------------------------------------------------------------
# LOAD OUTPUTS IF NEEDED
# ------------------------------------------------------------------

gold_gpkg = out_dir / "19_gold_deposit_dummy.gpkg"
gold_points_gpkg = out_dir / "19_gold_records_mrds_filtered.gpkg"

if "provinces_gold" not in globals():
    provinces_gold = gpd.read_file(gold_gpkg)

if "gold" not in globals():
    gold = gpd.read_file(gold_points_gpkg)

africa = gpd.read_file(africa_outline_file)

if africa.crs != provinces_gold.crs:
    africa = africa.to_crs(provinces_gold.crs)

if gold.crs != provinces_gold.crs:
    gold = gold.to_crs(provinces_gold.crs)

# ------------------------------------------------------------------
# MAP
# ------------------------------------------------------------------

var = "13_gold-deposit"

fig, ax = plt.subplots(figsize=(16, 20))

africa.plot(
    ax=ax,
    facecolor="none",
    edgecolor="lightgrey",
    linewidth=0.5
)

provinces_gold.plot(
    column=var,
    cmap="viridis",
    linewidth=0.4,
    edgecolor="black",
    legend=True,
    categorical=True,
    ax=ax
)

# Plots points of gold deposits  
""" gold.plot(
    ax=ax,
    color="gold",
    edgecolor="black",
    markersize=8,
    alpha=0.7
) """

ax.set_title("Gold deposit dummy by province")
ax.set_axis_off()

minx, miny, maxx, maxy = africa.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

out_file = map_output_dir / "19_gold_deposit_dummy_map.png"

plt.savefig(
    out_file,
    dpi=800,
    bbox_inches="tight"
)

plt.close()

print(f"Saved: {out_file}")

Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\13_gold\maps\19_gold_deposit_dummy_map.png
